In [3]:
# Original imports
from langdetect import detect
from deep_translator import GoogleTranslator

In [11]:
# Original Google Translator test
msg = "Hello ನಮಸ್ಕಾರ (how are you)"
print(detect(msg))
if(detect(msg) == 'en'):
    print(GoogleTranslator(source='en', target='kn').translate(msg))
else:
     print(msg)

en
ನಮಸ್ಕಾರ (ಹೇಗಿದ್ದೀರಿ)


In [5]:
# Another original test
(GoogleTranslator(source='en', target='kn').translate(msg))

'ಹಲೋ, ಹೇಗಿದ್ದೀಯಾ?  ನಮಸ್ಕಾರ! ಹೇಗಿದ್ದೀರಾ? ನೀವು 7 ನೇ ತರಗತಿಯ ವಿದ್ಯಾರ್ಥಿ ಎಂದು ನನಗೆ ತಿಳಿದಿದೆ. ನಾವು ಇಂದು'

---
## Azure Translator Implementation (New)
Below cells use Microsoft Azure Translator for more reliable translations

In [13]:
# Install required package if not already installed
# !pip install requests

import requests
import uuid
import time   # for latency measurement
import os

In [6]:
# Azure Translator Configuration
# Replace with your actual key from the Azure portal
import dotenv
AZURE_TRANSLATOR_ENDPOINT = "https://api.cognitive.microsofttranslator.com/"
AZURE_TRANSLATOR_REGION = "eastasia"
dotenv.load_dotenv(override=True, verbose=True)
AZURE_TRANSLATOR_KEY = os.getenv("AZURE_TRANSLATOR_KEY")

def translate_to_kannada(text):
    """
    Translates text from English to Kannada using Azure Translator.
    Also handles mixed language text.
    Prints latency (ms) after every API call.
    """
    path = '/translate'
    constructed_url = AZURE_TRANSLATOR_ENDPOINT + path
    
    params = {
        'api-version': '3.0',
        'from': 'en',
        'to': 'kn'
    }
    
    headers = {
        'Ocp-Apim-Subscription-Key': AZURE_TRANSLATOR_KEY,
        'Ocp-Apim-Subscription-Region': AZURE_TRANSLATOR_REGION,
        'Content-type': 'application/json',
        'X-ClientTraceId': str(uuid.uuid4())
    }
    
    body = [{
        'text': text
    }]
    
    try:
        _t0 = time.perf_counter()
        request = requests.post(constructed_url, params=params, headers=headers, json=body)
        _latency_ms = (time.perf_counter() - _t0) * 1000
        response = request.json()
        
        if request.status_code == 200:
            translated_text = response[0]['translations'][0]['text']
            detected_language = response[0]['detectedLanguage']['language'] if 'detectedLanguage' in response[0] else 'en'
            print(f"[Azure Translator] Latency: {_latency_ms:.1f} ms | '{text[:40]}...' -> '{translated_text[:40]}...'")
            return {
                'original': text,
                'translated': translated_text,
                'detected_language': detected_language,
                'latency_ms': round(_latency_ms, 2),
                'success': True
            }
        else:
            print(f"[Azure Translator] Latency: {_latency_ms:.1f} ms | FAILED (status {request.status_code})")
            return {
                'original': text,
                'error': response,
                'latency_ms': round(_latency_ms, 2),
                'success': False
            }
    except Exception as e:
        return {
            'original': text,
            'error': str(e),
            'success': False
        }

In [7]:
# Test 1: Pure English sentence
test_sentences = [
    "Hello, how are you?",
    "I am a student in class 7.",
    "Science is very interesting.",
    "What is photosynthesis?",
    "The capital of India is New Delhi."
]

print("=" * 80)
print("TEST 1: Pure English Sentences")
print("=" * 80)

for sentence in test_sentences:
    result = translate_to_kannada(sentence)
    if result['success']:
        print(f"\nOriginal: {result['original']}")
        print(f"Kannada:  {result['translated']}")
        print(f"Detected Language: {result['detected_language']}")
    else:
        print(f"\nError translating: {result['original']}")
        print(f"Error: {result['error']}")

TEST 1: Pure English Sentences



Original: Hello, how are you?
Kannada:  ಹಲೋ, ನೀವು ಹೇಗಿದ್ದೀರಿ?
Detected Language: en

Original: I am a student in class 7.
Kannada:  ನಾನು 7ನೇ ತರಗತಿಯ ವಿದ್ಯಾರ್ಥಿ.
Detected Language: en

Original: Science is very interesting.
Kannada:  ವಿಜ್ಞಾನ ಬಹಳ ಆಸಕ್ತಿದಾಯಕವಾಗಿದೆ.
Detected Language: en

Original: What is photosynthesis?
Kannada:  ದ್ಯುತಿಸಂಶ್ಲೇಷಣೆ ಎಂದರೇನು?
Detected Language: en

Original: The capital of India is New Delhi.
Kannada:  ಭಾರತದ ರಾಜಧಾನಿ ನವದೆಹಲಿ.
Detected Language: en


In [14]:
# Test 2: Mixed English-Kannada sentences
mixed_sentences = [
    "Hello, ನಮಸ್ಕಾರ! How are you doing today?",
    "I am studying science ಮತ್ತು mathematics in school",
    "The teacher said ಇಂದು we will learn about plants",
    "My favorite subject is ವಿಜ್ಞಾನ because it's interesting",
    "ನಾನು class 7ರ student ಆಗಿದ್ದೇನೆ",
    "Hello, how are you?  ನಮಸ್ಕಾರ! ಹೇಗಿದ್ದೀರಾ? ನೀವು ತರಗತಿ 7ರ ವಿದ್ಯಾರ್ಥಿ ಎಂದು ನನಗೆ ತಿಳಿದಿದೆ. ನಾವು ಇಂದು", 
    "Hello ನಮಸ್ಕಾರ (how are you)"
]

print("\n" + "=" * 80)
print("TEST 2: Mixed English-Kannada Sentences")
print("=" * 80)

for sentence in mixed_sentences:
    result = translate_to_kannada(sentence)
    if result['success']:
        print(f"\nOriginal: {result['original']}")
        print(f"Kannada:  {result['translated']}")
        print(f"Detected Language: {result['detected_language']}")
    else:
        print(f"\nError translating: {result['original']}")
        print(f"Error: {result['error']}")


TEST 2: Mixed English-Kannada Sentences

Original: Hello, ನಮಸ್ಕಾರ! How are you doing today?
Kannada:  ಹಲೋ, ನಮಸ್ಕಾರ! ನೀವು ಇಂದು ಹೇಗೆ ಇದ್ದೀರಿ?
Detected Language: en

Original: I am studying science ಮತ್ತು mathematics in school
Kannada:  ನಾನು ಶಾಲೆಯಲ್ಲಿ ವಿಜ್ಞಾನ ಮತ್ತು ಗಣಿತಶಾಸ್ತ್ರವನ್ನು ಅಧ್ಯಯನ ಮಾಡುತ್ತಿದ್ದೇನೆ
Detected Language: en

Original: The teacher said ಇಂದು we will learn about plants
Kannada:  ಶಿಕ್ಷಕರು ಇಂದು ನಾವು ಸಸ್ಯಗಳ ಬಗ್ಗೆ ಕಲಿಯುತ್ತೇವೆ ಎಂದು ಹೇಳಿದರು
Detected Language: en

Original: My favorite subject is ವಿಜ್ಞಾನ because it's interesting
Kannada:  ನನ್ನ ನೆಚ್ಚಿನ ವಿಷಯ ವಿಜ್ಞಾನ ಏಕೆಂದರೆ ಇದು ಆಸಕ್ತಿದಾಯಕವಾಗಿದೆ
Detected Language: en

Original: ನಾನು class 7ರ student ಆಗಿದ್ದೇನೆ
Kannada:  ನಾನು 7ನೇ ತರಗತಿಯ ವಿದ್ಯಾರ್ಥಿ ಆಗಿದ್ದೇನೆ
Detected Language: en

Original: Hello, how are you?  ನಮಸ್ಕಾರ! ಹೇಗಿದ್ದೀರಾ? ನೀವು ತರಗತಿ 7ರ ವಿದ್ಯಾರ್ಥಿ ಎಂದು ನನಗೆ ತಿಳಿದಿದೆ. ನಾವು ಇಂದು
Kannada:  ಹಲೋ ನೀವು ಹೇಗಿದ್ದೀರಿ?  ನಮಸ್ಕಾರ! ಹೇಗಿದ್ದೀರಾ? ನೀವು ತರಗತಿ 7ರ ವಿದ್ಯಾರ್ಥಿ ಎಂದು ನನಗೆ ತಿಳಿದಿದೆ. ನಾವು ಇಂದು
Detected Language: en

Original: Hello ನಮಸ್ಕಾ

In [21]:
# Advanced: Translate with auto-detection (no source language specified)
def translate_auto_detect(text, target_language='kn'):
    """
    Translates text to target language with automatic source language detection.
    Prints latency (ms) after every API call.
    """
    path = '/translate'
    constructed_url = AZURE_TRANSLATOR_ENDPOINT + path
    
    params = {
        'api-version': '3.0',
        'to': target_language
    }
    
    headers = {
        'Ocp-Apim-Subscription-Key': AZURE_TRANSLATOR_KEY,
        'Ocp-Apim-Subscription-Region': AZURE_TRANSLATOR_REGION,
        'Content-type': 'application/json',
        'X-ClientTraceId': str(uuid.uuid4())
    }
    
    body = [{
        'text': text
    }]
    
    try:
        _t0 = time.perf_counter()
        request = requests.post(constructed_url, params=params, headers=headers, json=body)
        _latency_ms = (time.perf_counter() - _t0) * 1000
        response = request.json()
        
        if request.status_code == 200:
            _translated = response[0]['translations'][0]['text']
            _detected = response[0]['detectedLanguage']['language']
            _conf = response[0]['detectedLanguage']['score']
            print(f"[Azure Auto-Detect] Latency: {_latency_ms:.1f} ms | detected={_detected} ({_conf:.2f}) | '{text[:40]}...' -> '{_translated[:40]}...'")
            return {
                'original': text,
                'translated': _translated,
                'detected_language': _detected,
                'confidence': _conf,
                'latency_ms': round(_latency_ms, 2),
                'success': True
            }
        else:
            print(f"[Azure Auto-Detect] Latency: {_latency_ms:.1f} ms | FAILED (status {request.status_code})")
            return {'original': text, 'error': response, 'latency_ms': round(_latency_ms, 2), 'success': False}
    except Exception as e:
        return {'original': text, 'error': str(e), 'success': False}

# Test auto-detection
print("\n" + "=" * 80)
print("TEST 3: Auto Language Detection")
print("=" * 80)

auto_test = [
    "Hello world",
    "\u0ca8\u0cae\u0cb8\u0ccd\u0c95\u0cbe\u0cb0 \u0cb5\u0cbf\u0cb6\u0ccd\u0cb5",
    "Bonjour le monde",
    "\u0928\u092e\u0938\u094d\u0924\u0947 \u0926\u0941\u0928\u093f\u092f\u093e"
]

for text in auto_test:
    result = translate_auto_detect(text)
    if result['success']:
        print(f"\nOriginal: {result['original']}")
        print(f"Detected: {result['detected_language']} (confidence: {result['confidence']:.2f})")
        print(f"Kannada:  {result['translated']}")


TEST 3: Auto Language Detection

Original: Hello world
Detected: en (confidence: 0.90)
Kannada:  ಹಲೋ ಪ್ರಪಂಚ

Original: ನಮಸ್ಕಾರ ವಿಶ್ವ
Detected: kn (confidence: 1.00)
Kannada:  ನಮಸ್ಕಾರ ವಿಶ್ವ

Original: Bonjour le monde
Detected: fr (confidence: 0.98)
Kannada:  ಹಲೋ ಪ್ರಪಂಚ

Original: नमस्ते दुनिया
Detected: hi (confidence: 0.84)
Kannada:  ಹಲೋ ಪ್ರಪಂಚ


---
## Smart Translation with Local Language Detection
Using `langdetect` to check language locally BEFORE calling Azure API (saves API calls!)


In [16]:
from langdetect import detect, DetectorFactory
# Set seed for consistent results
DetectorFactory.seed = 0

def smart_translate_to_kannada(text, confidence_threshold=0.7):
    """
    Smart translation that:
    1. Detects language locally (no API call)
    2. Only calls Azure if text is NOT already Kannada
    3. Tracks whether API call was made
    
    Args:
        text: Text to translate
        confidence_threshold: Minimum confidence to trust detection (0-1)
    
    Returns:
        dict with translation result and API call status
    """
    stats = {
        'original': text,
        'api_call_made': False,
        'detected_language': None,
        'action_taken': None
    }
    
    try:
        # LOCAL detection - no API call!
        detected_lang = detect(text)
        stats['detected_language'] = detected_lang
        
        # If already Kannada, skip translation
        if detected_lang == 'kn':
            stats['translated'] = text
            stats['action_taken'] = 'No translation needed - already Kannada'
            stats['api_call_made'] = False
            return stats
        
        # If English or other language, translate via Azure
        stats['action_taken'] = f'Translating from {detected_lang} to Kannada via Azure API'
        stats['api_call_made'] = True
        
        # Call Azure API
        path = '/translate'
        constructed_url = AZURE_TRANSLATOR_ENDPOINT + path
        
        params = {
            'api-version': '3.0',
            'from': detected_lang,
            'to': 'kn'
        }
        
        headers = {
            'Ocp-Apim-Subscription-Key': AZURE_TRANSLATOR_KEY,
            'Ocp-Apim-Subscription-Region': AZURE_TRANSLATOR_REGION,
            'Content-type': 'application/json',
            'X-ClientTraceId': str(uuid.uuid4())
        }
        
        body = [{'text': text}]
        
        request = requests.post(constructed_url, params=params, headers=headers, json=body)
        response = request.json()
        
        if request.status_code == 200:
            stats['translated'] = response[0]['translations'][0]['text']
            stats['success'] = True
        else:
            stats['translated'] = text
            stats['error'] = response
            stats['success'] = False
            
    except Exception as e:
        # If detection fails, default to translating
        stats['action_taken'] = f'Detection failed ({str(e)}), defaulting to Azure translation'
        stats['api_call_made'] = True
        
        # Call Azure API as fallback
        try:
            result = translate_to_kannada(text)
            stats['translated'] = result['translated']
            stats['success'] = result['success']
        except:
            stats['translated'] = text
            stats['success'] = False
    
    return stats

print("✅ Smart translation function loaded!")
print("💡 This function detects language LOCALLY before calling Azure API")


✅ Smart translation function loaded!
💡 This function detects language LOCALLY before calling Azure API


In [17]:
# TEST 4: Smart Translation - API Call Optimization
# This test demonstrates API call savings

test_cases = [
    # Pure English - WILL call API
    ("Hello, how are you today?", "Pure English"),
    ("What is the capital of India?", "Pure English"),
    ("Science is very interesting to learn", "Pure English"),
    
    # Pure Kannada - WILL NOT call API (saves API call!)
    ("ನಮಸ್ಕಾರ, ನೀವು ಹೇಗಿದ್ದೀರಿ?", "Pure Kannada"),
    ("ನಾನು ತರಗತಿ 7ರ ವಿದ್ಯಾರ್ಥಿ", "Pure Kannada"),
    ("ವಿಜ್ಞಾನ ಬಹಳ ಆಸಕ್ತಿದಾಯಕವಾಗಿದೆ", "Pure Kannada"),
    
    # Mixed with more English - WILL call API
    ("Hello ನಮಸ್ಕಾರ how are you?", "Mixed (60% English)"),
    ("I am studying ವಿಜ್ಞಾನ today", "Mixed (70% English)"),
    
    # Mixed with more Kannada - depends on detection
    ("ನಮಸ್ಕಾರ hello ಹೇಗಿದ್ದೀರಾ", "Mixed (70% Kannada)"),
    ("ನಾನು studying mathematics ಮತ್ತು science", "Mixed (60% Kannada)"),
    
    # Edge cases
    ("ನಮಸ್ಕಾರ! ಹೇಗಿದ್ದೀರಾ? ನೀವು ತರಗತಿ 7ರ ವಿದ್ಯಾರ್ಥಿ ಎಂದು ನನಗೆ ತಿಳಿದಿದೆ.", "Long Kannada sentence"),
    ("The water cycle includes evaporation, condensation, and precipitation.", "Technical English"),
]

print("=" * 100)
print("TEST 4: SMART TRANSLATION - API CALL OPTIMIZATION TEST")
print("=" * 100)
print("\n📊 API Call Tracker:")
print("-" * 100)

api_calls_made = 0
api_calls_saved = 0

for text, description in test_cases:
    result = smart_translate_to_kannada(text)
    
    if result['api_call_made']:
        api_calls_made += 1
        call_status = "🔴 API CALL MADE"
    else:
        api_calls_saved += 1
        call_status = "✅ API CALL SAVED"
    
    print(f"\n{call_status}")
    print(f"Type: {description}")
    print(f"Original:  {result['original'][:80]}...")
    print(f"Detected:  {result['detected_language']}")
    print(f"Action:    {result['action_taken']}")
    print(f"Result:    {result['translated'][:80]}...")
    print("-" * 100)

print(f"\n📈 SUMMARY:")
print(f"   🔴 API Calls Made: {api_calls_made}")
print(f"   ✅ API Calls Saved: {api_calls_saved}")
print(f"   💰 Savings: {api_calls_saved}/{len(test_cases)} = {(api_calls_saved/len(test_cases)*100):.1f}%")
print("=" * 100)


TEST 4: SMART TRANSLATION - API CALL OPTIMIZATION TEST

📊 API Call Tracker:
----------------------------------------------------------------------------------------------------

🔴 API CALL MADE
Type: Pure English
Original:  Hello, how are you today?...
Detected:  so
Action:    Translating from so to Kannada via Azure API
Result:    ಹಲೋ ನೀವು ಇಂದು ಹೇಗಿದ್ದೀರಿ?...
----------------------------------------------------------------------------------------------------

🔴 API CALL MADE
Type: Pure English
Original:  What is the capital of India?...
Detected:  en
Action:    Translating from en to Kannada via Azure API
Result:    ಭಾರತದ ರಾಜಧಾನಿ ಯಾವುದು?...
----------------------------------------------------------------------------------------------------

🔴 API CALL MADE
Type: Pure English
Original:  Science is very interesting to learn...
Detected:  en
Action:    Translating from en to Kannada via Azure API
Result:    ವಿಜ್ಞಾನವು ಕಲಿಯಲು ತುಂಬಾ ಆಸಕ್ತಿದಾಯಕವಾಗಿದೆ...
--------------------------------------

---
## Improved Smart Translation - Detects ANY English
This version checks for English characters directly, ensuring even mixed text gets translated


In [19]:
import re

def improved_smart_translate(text):
    """
    Improved smart translation that:
    1. Checks for ANY English characters locally (no API call)
    2. Only skips API if text is PURELY Kannada (no English at all)
    3. Makes API call if even a single English word exists
    
    Args:
        text: Text to translate
    
    Returns:
        dict with translation result and API call status
    """
    stats = {
        'original': text,
        'api_call_made': False,
        'contains_english': False,
        'action_taken': None
    }
    
    try:
        # LOCAL CHECK: Does text contain ANY English characters?
        # Matches any English letters (a-z, A-Z)
        has_english = bool(re.search(r'[a-zA-Z]', text))
        stats['contains_english'] = has_english
        
        # If NO English characters at all, skip translation
        if not has_english:
            stats['translated'] = text
            stats['action_taken'] = 'No translation needed - pure Kannada, no English detected'
            stats['api_call_made'] = False
            stats['success'] = True
            return stats
        
        # If ANY English found, translate via Azure
        stats['action_taken'] = f'English characters detected - calling Azure API for translation'
        stats['api_call_made'] = True
        
        # Call Azure API
        path = '/translate'
        constructed_url = AZURE_TRANSLATOR_ENDPOINT + path
        
        # KEY FIX: Use 'from': 'en' to force Azure to treat it as English
        # This ensures mixed text gets translated properly
        params = {
            'api-version': '3.0',
            'from': 'en',  # Force English source - Azure will translate English parts
            'to': 'kn'
        }
        
        headers = {
            'Ocp-Apim-Subscription-Key': AZURE_TRANSLATOR_KEY,
            'Ocp-Apim-Subscription-Region': AZURE_TRANSLATOR_REGION,
            'Content-type': 'application/json',
            'X-ClientTraceId': str(uuid.uuid4())
        }
        
        body = [{'text': text}]
        
        request = requests.post(constructed_url, params=params, headers=headers, json=body)
        response = request.json()
        
        if request.status_code == 200:
            stats['translated'] = response[0]['translations'][0]['text']
            # When 'from' is specified, Azure doesn't return detectedLanguage
            stats['detected_language'] = response[0]['detectedLanguage']['language'] if 'detectedLanguage' in response[0] else 'en'
            stats['success'] = True
        else:
            stats['translated'] = text
            stats['error'] = response
            stats['success'] = False
            
    except Exception as e:
        # If anything fails, default to original text
        stats['action_taken'] = f'Error occurred ({str(e)}), returning original text'
        stats['translated'] = text
        stats['success'] = False
    
    return stats

print("✅ Improved smart translation function loaded!")
print("💡 This function checks for English characters locally - ANY English triggers API call")


✅ Improved smart translation function loaded!
💡 This function checks for English characters locally - ANY English triggers API call


In [20]:
# TEST 5: Improved Smart Translation - English Detection Test
# This test shows that even a SINGLE English word triggers API call

test_cases_v2 = [
    # Pure English - WILL call API
    ("Hello, how are you today?", "Pure English"),
    ("What is the capital of India?", "Pure English"),
    ("Science is very interesting to learn", "Pure English"),
    
    # Pure Kannada (NO English at all) - WILL NOT call API (saves API call!)
    ("ನಮಸ್ಕಾರ, ನೀವು ಹೇಗಿದ್ದೀರಿ?", "Pure Kannada - no English"),
    ("ನಾನು ತರಗತಿ ೭ರ ವಿದ್ಯಾರ್ಥಿ", "Pure Kannada - no English"),
    ("ವಿಜ್ಞಾನ ಬಹಳ ಆಸಕ್ತಿದಾಯಕವಾಗಿದೆ", "Pure Kannada - no English"),
    
    # Mixed with MORE English - WILL call API
    ("Hello ನಮಸ್ಕಾರ (how are you)?", "Mixed (60% English)"),
    ("I am studying ವಿಜ್ಞಾನ today", "Mixed (70% English)"),
    
    # Mixed with MORE Kannada but has English - WILL call API (this is your case!)
    ("ನಮಸ್ಕಾರ hello ಹೇಗಿದ್ದೀರಾ", "Mixed (70% Kannada, but has 'hello')"),
    ("ನಾನು studying mathematics ಮತ್ತು science", "Mixed (60% Kannada, has English words)"),
    
    # Edge cases
    ("ನಮಸ್ಕಾರ! ಹೇಗಿದ್ದೀರಾ? ನೀವು ತರಗತಿ ೭ರ ವಿದ್ಯಾರ್ಥಿ ಎಂದು ನನಗೆ ತಿಳಿದಿದೆ.", "Long Kannada - NO English"),
    ("The water cycle includes evaporation, condensation, and precipitation.", "Technical English"),
    
    # Tricky: Numbers and punctuation only (no English) - WILL NOT call API
    ("೧೨೩೪೫! ನಮಸ್ಕಾರ? ಹೇಗಿದ್ದೀರಾ.", "Kannada with numbers/punctuation"),
    
    # Tricky: Single English letter - WILL call API
    ("ನಮಸ್ಕಾರ a ಹೇಗಿದ್ದೀರಾ", "Mostly Kannada with single English letter 'a'"),
]

print("=" * 100)
print("TEST 5: IMPROVED SMART TRANSLATION - ENGLISH CHARACTER DETECTION TEST")
print("=" * 100)
print("\n📊 API Call Tracker (ANY English = API Call):")
print("-" * 100)

api_calls_made_v2 = 0
api_calls_saved_v2 = 0

for text, description in test_cases_v2:
    result = improved_smart_translate(text)
    
    if result['api_call_made']:
        api_calls_made_v2 += 1
        call_status = "🔴 API CALL MADE"
    else:
        api_calls_saved_v2 += 1
        call_status = "✅ API CALL SAVED"
    
    print(f"\n{call_status}")
    print(f"Type: {description}")
    print(f"Original:  {result['original'][:80]}...")
    print(f"Has English? {result['contains_english']}")
    print(f"Action:    {result['action_taken']}")
    print(f"Result:    {result.get('translated', 'N/A')[:80]}...")
    print("-" * 100)

print(f"\n📈 SUMMARY:")
print(f"   🔴 API Calls Made: {api_calls_made_v2}")
print(f"   ✅ API Calls Saved: {api_calls_saved_v2}")
print(f"   💰 Savings: {api_calls_saved_v2}/{len(test_cases_v2)} = {(api_calls_saved_v2/len(test_cases_v2)*100):.1f}%")
print(f"\n💡 KEY INSIGHT: Only PURE Kannada (no English letters) skips API call!")
print("=" * 100)


TEST 5: IMPROVED SMART TRANSLATION - ENGLISH CHARACTER DETECTION TEST

📊 API Call Tracker (ANY English = API Call):
----------------------------------------------------------------------------------------------------

🔴 API CALL MADE
Type: Pure English
Original:  Hello, how are you today?...
Has English? True
Action:    English characters detected - calling Azure API for translation
Result:    ಹಲೋ ನೀವು ಇಂದು ಹೇಗಿದ್ದೀರಿ?...
----------------------------------------------------------------------------------------------------

🔴 API CALL MADE
Type: Pure English
Original:  What is the capital of India?...
Has English? True
Action:    English characters detected - calling Azure API for translation
Result:    ಭಾರತದ ರಾಜಧಾನಿ ಯಾವುದು?...
----------------------------------------------------------------------------------------------------

🔴 API CALL MADE
Type: Pure English
Original:  Science is very interesting to learn...
Has English? True
Action:    English characters detected - calling Azure API

In [26]:
# TEST 6: Test the production function from shared_utils.py
import sys
sys.path.append('..')  # Add parent directory to path

from utils.shared_utils import translate_to_kannada_azure

print("=" * 100)
print("TEST 6: TESTING PRODUCTION FUNCTION FROM shared_utils.py")
print("=" * 100)

# Test with the same mixed sentence that was failing before
test_sentence = "ನಮಸ್ಕಾರ! ನೀವು ಕಲಿಯಲು ಸಿದ್ಧರಿದ್ದೀರಾ? ನಾವು ಇಂದು ಉಷ್ಣ ವರ್ಗಾವಣೆ (ಹೀಟ್ ಟ್ರಾನ್ಸ್ಫರ್) ಯ ಒಂದು ಪ್ರಮುಖ ವಿಧಾನವಾದ 'ಸಂವಹನ' (Conduction) ಬಗ್ಗೆ ಕಲಿಯೋಣ"

print(f"\n📝 Original: {test_sentence}")
print("🔄 Calling translate_to_kannada_azure()...")

result = translate_to_kannada_azure(test_sentence)

print(f"✅ Translated: {result}")
print("\n" + "=" * 100)

print("\n💡 Production function uses same strategy: 'from': 'en' to force translation!")
print("=" * 100)


TEST 6: TESTING PRODUCTION FUNCTION FROM shared_utils.py

📝 Original: ನಮಸ್ಕಾರ! ನೀವು ಕಲಿಯಲು ಸಿದ್ಧರಿದ್ದೀರಾ? ನಾವು ಇಂದು ಉಷ್ಣ ವರ್ಗಾವಣೆ (ಹೀಟ್ ಟ್ರಾನ್ಸ್ಫರ್) ಯ ಒಂದು ಪ್ರಮುಖ ವಿಧಾನವಾದ 'ಸಂವಹನ' (Conduction) ಬಗ್ಗೆ ಕಲಿಯೋಣ
🔄 Calling translate_to_kannada_azure()...
✅ Translated to Kannada (detected: unknown): ನಮಸ್ಕಾರ! ನೀವು ಕಲಿಯಲು ಸಿದ್ಧರಿದ್ದೀರಾ? ನಾವು ಇಂದು ಉಷ್ಣ...
✅ Translated: ನಮಸ್ಕಾರ! ನೀವು ಕಲಿಯಲು ಸಿದ್ಧರಿದ್ದೀರಾ? ನಾವು ಇಂದು ಉಷ್ಣ ವರ್ಗಾವಣೆ (ಹೀಟ್ ಟ್ರಾನ್ಸ್ಫರ್) ಯ ಒಂದು ಪ್ರಮುಖ ವಿಧಾನವಾದ 'ಸಂವಹನ' (Conduction) ಬಗ್ಗೆ ಕಲಿಯೋಣ


💡 Production function uses same strategy: 'from': 'en' to force translation!


---
## Deep Translator - GoogleTranslator Implementation
Using `deep_translator` library with GoogleTranslator for Kannada translation + Regex-based English detection

In [26]:
# Import deep_translator and GoogleTranslator
from deep_translator import GoogleTranslator
import re

print("✅ GoogleTranslator from deep_translator imported successfully!")
print("📚 Library: deep_translator")
print("🌐 Translator: Google Translate API (free)")
print("💡 Features: No API keys needed, supports 100+ languages")

✅ GoogleTranslator from deep_translator imported successfully!
📚 Library: deep_translator
🌐 Translator: Google Translate API (free)
💡 Features: No API keys needed, supports 100+ languages


In [27]:
def translate_to_kannada_google(text):
    """
    Translates text from English to Kannada using GoogleTranslator from deep_translator.
    Prints latency (ms) after every call.

    Args:
        text: Text to translate

    Returns:
        dict with translation result including latency_ms
    """
    try:
        _t0 = time.perf_counter()
        translated_text = GoogleTranslator(source='en', target='kn').translate(text)
        _latency_ms = (time.perf_counter() - _t0) * 1000
        print(f"[GoogleTranslator] Latency: {_latency_ms:.1f} ms | '{text[:40]}...' -> '{translated_text[:40]}...'")
        return {
            'original': text,
            'translated': translated_text,
            'latency_ms': round(_latency_ms, 2),
            'success': True,
            'method': 'GoogleTranslator (deep_translator)'
        }
    except Exception as e:
        return {
            'original': text,
            'error': str(e),
            'success': False,
            'method': 'GoogleTranslator (deep_translator)'
        }

print("Basic GoogleTranslator translation function loaded!")

Basic GoogleTranslator translation function loaded!


In [28]:
# TEST 7: GoogleTranslator - Pure English Sentences
test_sentences_google = [
    "Hello, how are you?",
    "I am a student in class 7.",
    "Science is very interesting.",
    "What is photosynthesis?",
    "The capital of India is New Delhi."
]

print("=" * 100)
print("TEST 7: GoogleTranslator - Pure English Sentences")
print("=" * 100)

for sentence in test_sentences_google:
    result = translate_to_kannada_google(sentence)
    if result['success']:
        print(f"\n✅ Success")
        print(f"Original:    {result['original']}")
        print(f"Kannada:     {result['translated']}")
    else:
        print(f"\n❌ Error")
        print(f"Original:    {result['original']}")
        print(f"Error:       {result['error']}")

print("=" * 100)

TEST 7: GoogleTranslator - Pure English Sentences


[GoogleTranslator] Latency: 174.9 ms | 'Hello, how are you?...' -> 'ಹಲೋ, ಹೇಗಿದ್ದೀಯಾ?...'

✅ Success
Original:    Hello, how are you?
Kannada:     ಹಲೋ, ಹೇಗಿದ್ದೀಯಾ?
[GoogleTranslator] Latency: 299.5 ms | 'I am a student in class 7....' -> 'ನಾನು 7ನೇ ತರಗತಿ ವಿದ್ಯಾರ್ಥಿ....'

✅ Success
Original:    I am a student in class 7.
Kannada:     ನಾನು 7ನೇ ತರಗತಿ ವಿದ್ಯಾರ್ಥಿ.
[GoogleTranslator] Latency: 242.2 ms | 'Science is very interesting....' -> 'ವಿಜ್ಞಾನವು ತುಂಬಾ ಆಸಕ್ತಿದಾಯಕವಾಗಿದೆ....'

✅ Success
Original:    Science is very interesting.
Kannada:     ವಿಜ್ಞಾನವು ತುಂಬಾ ಆಸಕ್ತಿದಾಯಕವಾಗಿದೆ.
[GoogleTranslator] Latency: 154.2 ms | 'What is photosynthesis?...' -> 'ದ್ಯುತಿಸಂಶ್ಲೇಷಣೆ ಎಂದರೇನು?...'

✅ Success
Original:    What is photosynthesis?
Kannada:     ದ್ಯುತಿಸಂಶ್ಲೇಷಣೆ ಎಂದರೇನು?
[GoogleTranslator] Latency: 191.3 ms | 'The capital of India is New Delhi....' -> 'ಭಾರತದ ರಾಜಧಾನಿ ನವದೆಹಲಿ....'

✅ Success
Original:    The capital of India is New Delhi.
Kannada:     ಭಾರತದ ರಾಜಧಾನಿ ನವದೆಹಲಿ.


In [29]:
# TEST 8: GoogleTranslator - Mixed English-Kannada Sentences
mixed_sentences_google = [
    "Hello, ನಮಸ್ಕಾರ! How are you doing today?",
    "I am studying science ಮತ್ತು mathematics in school",
    "The teacher said ಇಂದು we will learn about plants",
    "My favorite subject is ವಿಜ್ಞಾನ because it's interesting",
    "ನಾನು class 7ರ student ಆಗಿದ್ದೇನೆ"
]

print("\n" + "=" * 100)
print("TEST 8: GoogleTranslator - Mixed English-Kannada Sentences")
print("=" * 100)

for sentence in mixed_sentences_google:
    result = translate_to_kannada_google(sentence)
    if result['success']:
        print(f"\n✅ Success")
        print(f"Original:    {result['original'][:80]}")
        print(f"Kannada:     {result['translated'][:80]}")
    else:
        print(f"\n❌ Error")
        print(f"Original:    {result['original'][:80]}")
        print(f"Error:       {result['error']}")

print("=" * 100)


TEST 8: GoogleTranslator - Mixed English-Kannada Sentences
[GoogleTranslator] Latency: 357.5 ms | 'Hello, ನಮಸ್ಕಾರ! How are you doing today?...' -> 'ನಮಸ್ಕಾರ, ನಮಸ್ಕಾರ! ಇವತ್ತು ಹೇಗಿದ್ದೀಯ?...'

✅ Success
Original:    Hello, ನಮಸ್ಕಾರ! How are you doing today?
Kannada:     ನಮಸ್ಕಾರ, ನಮಸ್ಕಾರ! ಇವತ್ತು ಹೇಗಿದ್ದೀಯ?
[GoogleTranslator] Latency: 353.4 ms | 'I am studying science ಮತ್ತು mathematics ...' -> 'ನಾನು ಶಾಲೆಯಲ್ಲಿ ವಿಜ್ಞಾನ ಮತ್ತು ಗಣಿತವನ್ನು ಓ...'

✅ Success
Original:    I am studying science ಮತ್ತು mathematics in school
Kannada:     ನಾನು ಶಾಲೆಯಲ್ಲಿ ವಿಜ್ಞಾನ ಮತ್ತು ಗಣಿತವನ್ನು ಓದುತ್ತಿದ್ದೇನೆ
[GoogleTranslator] Latency: 176.3 ms | 'The teacher said ಇಂದು we will learn abou...' -> 'ಇಂದು ನಾವು ಸಸ್ಯಗಳ ಬಗ್ಗೆ ಕಲಿಯುತ್ತೇವೆ ಎಂದು ...'

✅ Success
Original:    The teacher said ಇಂದು we will learn about plants
Kannada:     ಇಂದು ನಾವು ಸಸ್ಯಗಳ ಬಗ್ಗೆ ಕಲಿಯುತ್ತೇವೆ ಎಂದು ಶಿಕ್ಷಕರು ಹೇಳಿದರು
[GoogleTranslator] Latency: 171.2 ms | 'My favorite subject is ವಿಜ್ಞಾನ because i...' -> 'ನನ್ನ ನೆಚ್ಚಿನ ವಿಷಯ ವಿಜ್ಞಾನ ಏಕೆಂದರೆ ಇದು ಆಸ...'

✅ Success
O

In [30]:
def detect_english_with_regex(text):
    """
    Detects if text contains ANY English letter using regex.
    
    Simple rule: If even 1 English letter [a-zA-Z] exists, has_english = True
    
    Args:
        text: Text to analyze
        
    Returns:
        dict with detection results
    """
    stats = {
        'original_text': text,
        'has_english': False,
        'english_letters': []
    }
    
    # Pattern: Any English letter (a-z, A-Z)
    pattern = r'[a-zA-Z]'
    matches = re.findall(pattern, text)
    
    if matches:
        stats['has_english'] = True
        stats['english_letters'] = matches[:10]  # Show first 10 English letters found
    
    return stats

print("✅ English regex detection function loaded!")
print("📊 Simple rule: If ANY English letter exists, translate!")
print("💡 No confidence intervals - just: English letter found? Yes/No")

✅ English regex detection function loaded!
📊 Simple rule: If ANY English letter exists, translate!
💡 No confidence intervals - just: English letter found? Yes/No


In [31]:
# TEST 9: Regex English Detection Test - Simple Binary (Has English or Not)
detection_test_cases = [
    # Pure English - SHOULD detect
    ("Hello world", True),
    ("I am a student", True),
    ("What is photosynthesis?", True),
    
    # Pure Kannada (NO English) - should NOT detect
    ("ನಮಸ್ಕಾರ", False),
    ("ನಾನು ತರಗತಿ ೭ರ ವಿದ್ಯಾರ್ಥಿ", False),
    ("ವಿಜ್ಞಾನ ಬಹಳ ಆಸಕ್ತಿದಾಯಕವಾಗಿದೆ", False),
    
    # Mixed - SHOULD detect (has English)
    ("Hello ನಮಸ್ಕಾರ", True),
    ("I am studying ವಿಜ್ಞಾನ", True),
    ("ನಮಸ್ಕಾರ hello ಹೇಗಿದ್ದೀರಾ", True),
    ("ನಾನು studying mathematics ಮತ್ತು science", True),
    
    # Edge cases
    ("೧೨೩ ನಮಸ್ಕಾರ! ಹೇಗಿದ್ದೀರಾ?", False),  # Kannada numbers and text, no English
    ("COVID19 ತೆರೆಯುತ್ತಿದೆ", True),  # Has English letters in COVID
    ("Hello ನಮಸ್ಕಾರ (how are you)? ಹೇಗಿದ್ದೀರಾ?", True),  # Mixed with parentheses
    ("ನಾನು a ವಿದ್ಯಾರ್ಥಿ", True),  # Single English letter 'a'
]

print("\n" + "=" * 100)
print("TEST 9: Regex English Detection - Binary (Has English or Not)")
print("=" * 100)
print()

correct_detections = 0
for text, should_have_english in detection_test_cases:
    detection = detect_english_with_regex(text)
    is_correct = detection['has_english'] == should_have_english
    correct_detections += is_correct
    
    status = "✅ CORRECT" if is_correct else "❌ INCORRECT"
    detected = "English FOUND" if detection['has_english'] else "NO English"
    expected = "should have English" if should_have_english else "should NOT have English"
    
    print(f"{status}")
    print(f"Text: {text[:70]}")
    print(f"Result: {detected} | Expected: {expected}")
    if detection['english_letters']:
        print(f"English letters found: {detection['english_letters']}")
    print("-" * 100)

print(f"\n📊 Accuracy: {correct_detections}/{len(detection_test_cases)} = {(correct_detections/len(detection_test_cases)*100):.1f}%")
print("=" * 100)


TEST 9: Regex English Detection - Binary (Has English or Not)

✅ CORRECT
Text: Hello world
Result: English FOUND | Expected: should have English
English letters found: ['H', 'e', 'l', 'l', 'o', 'w', 'o', 'r', 'l', 'd']
----------------------------------------------------------------------------------------------------
✅ CORRECT
Text: I am a student
Result: English FOUND | Expected: should have English
English letters found: ['I', 'a', 'm', 'a', 's', 't', 'u', 'd', 'e', 'n']
----------------------------------------------------------------------------------------------------
✅ CORRECT
Text: What is photosynthesis?
Result: English FOUND | Expected: should have English
English letters found: ['W', 'h', 'a', 't', 'i', 's', 'p', 'h', 'o', 't']
----------------------------------------------------------------------------------------------------
✅ CORRECT
Text: ನಮಸ್ಕಾರ
Result: NO English | Expected: should NOT have English
-----------------------------------------------------------------------

In [32]:
def smart_google_translate(text):
    """
    Smart GoogleTranslator that:
    1. Uses REGEX to check if text contains ANY English letter [a-zA-Z]
    2. If ANY English letter found -> TRANSLATE (prints latency)
    3. If NO English letters found -> Return original (pure Kannada, no translation needed)
    4. Tracks whether translation was performed

    Args:
        text: Text to potentially translate

    Returns:
        dict with translation result, optimization status, and latency_ms
    """
    stats = {
        'original': text,
        'translation_performed': False,
        'has_english_detected': False,
        'action_taken': None,
        'latency_ms': 0.0,
        'success': False
    }

    # Step 1: Detect English using regex (LOCAL, NO API CALL)
    english_detection = detect_english_with_regex(text)
    stats['has_english_detected'] = english_detection['has_english']
    stats['english_letters_found'] = len(english_detection['english_letters'])

    # Step 2: If NO English letters found, skip translation
    if not english_detection['has_english']:
        stats['translated'] = text
        stats['action_taken'] = 'No English letters found - Pure Kannada! Skip translation (save call!)'
        stats['translation_performed'] = False
        stats['success'] = True
        return stats

    # Step 3: English letter(s) detected - perform translation
    stats['action_taken'] = f"English letter(s) detected ({stats['english_letters_found']} found) - Translating..."

    try:
        _t0 = time.perf_counter()
        translated_text = GoogleTranslator(source='en', target='kn').translate(text)
        _latency_ms = (time.perf_counter() - _t0) * 1000
        stats['translated'] = translated_text
        stats['translation_performed'] = True
        stats['latency_ms'] = round(_latency_ms, 2)
        stats['success'] = True
        print(f"[GoogleTranslator] Latency: {_latency_ms:.1f} ms | '{text[:40]}...' -> '{translated_text[:40]}...'")
    except Exception as e:
        stats['error'] = str(e)
        stats['translated'] = text
        stats['action_taken'] = f"Translation error: {str(e)}"
        stats['translation_performed'] = False
        stats['success'] = False

    return stats

print("Smart GoogleTranslator with Binary English Detection loaded!")
print("Rule: Even 1 English letter = Translate")
print("Pure Kannada (0 English letters) = Skip translation")

Smart GoogleTranslator with Binary English Detection loaded!
Rule: Even 1 English letter = Translate
Pure Kannada (0 English letters) = Skip translation


In [33]:
# TEST 10: Smart GoogleTranslator - Binary English Detection (Any Letter = Translate)
smart_google_test_cases = [
    # Pure English - HAS English letters -> WILL translate
    ("Hello, how are you today?", "Pure English"),
    ("What is the capital of India?", "Pure English"),
    ("Science is very interesting to learn", "Pure English"),
    
    # Pure Kannada (0 English letters) - WILL NOT translate (saves call!)
    ("ನಮಸ್ಕಾರ, ನೀವು ಹೇಗಿದ್ದೀರಿ?", "Pure Kannada - 0 English"),
    ("ನಾನು ತರಗತಿ ೭ರ ವಿದ್ಯಾರ್ಥಿ", "Pure Kannada - 0 English"),
    ("ವಿಜ್ಞಾನ ಬಹಳ ಆಸಕ್ತಿದಾಯಕವಾಗಿದೆ", "Pure Kannada - 0 English"),
    
    # Mixed with MORE English - HAS English letters -> WILL translate
    ("Hello ನಮಸ್ಕಾರ (how are you)?", "Mixed - has English"),
    ("I am studying ವಿಜ್ಞಾನ today", "Mixed - has English"),
    
    # Mixed with MORE Kannada BUT HAS English - HAS English letters -> WILL translate
    ("ನಮಸ್ಕಾರ hello ಹೇಗಿದ್ದೀರಾ", "Mixed - has 'hello'"),
    ("ನಾನು studying mathematics ಮತ್ತು science", "Mixed - has English words"),
    
    # Edge cases
    ("ನಮಸ್ಕಾರ! ಹೇಗಿದ್ದೀರಾ? ನೀವು ತರಗತಿ ೭ರ ವಿದ್ಯಾರ್ಥಿ ಎಂದು ನನಗೆ ತಿಳಿದಿದೆ.", "Long Kannada - 0 English"),
    ("The water cycle includes evaporation, condensation, and precipitation.", "Technical English"),
    ("ನಾನು a student ಆಗಿದ್ದೇನೆ", "Mixed - even single 'a' triggers translate"),
    ("೧೨೩ ನಮಸ್ಕಾರ! ಹೇಗಿದ್ದೀರಾ?", "Kannada numbers only - 0 English"),
]

print("\n" + "=" * 100)
print("TEST 10: Smart GoogleTranslator - Binary Detection (Any English Letter = Translate)")
print("=" * 100)
print()

translations_performed = 0
translations_saved = 0

for text, description in smart_google_test_cases:
    result = smart_google_translate(text)
    
    if result['translation_performed']:
        translations_performed += 1
        call_status = "🔴 TRANSLATED"
    else:
        translations_saved += 1
        call_status = "✅ SKIPPED"
    
    print(f"{call_status}")
    print(f"Type: {description}")
    print(f"Original:       {result['original'][:65]}")
    print(f"English found?  {result['has_english_detected']}")
    print(f"Action:         {result['action_taken']}")
    print(f"Result:         {result['translated'][:65]}")
    print("-" * 100)

print(f"\n📈 OPTIMIZATION SUMMARY:")
print(f"   🔴 Translations Performed: {translations_performed}")
print(f"   ✅ Translations Skipped: {translations_saved}")
print(f"   💰 Savings: {translations_saved}/{len(smart_google_test_cases)} = {(translations_saved/len(smart_google_test_cases)*100):.1f}%")
print(f"\n💡 RULE: If regex finds ANY [a-zA-Z] letter, translate. Otherwise skip!")
print("=" * 100)


TEST 10: Smart GoogleTranslator - Binary Detection (Any English Letter = Translate)

[GoogleTranslator] Latency: 471.5 ms | 'Hello, how are you today?...' -> 'ಹಲೋ, ಇವತ್ತು ಹೇಗಿದ್ದೀಯ?...'
🔴 TRANSLATED
Type: Pure English
Original:       Hello, how are you today?
English found?  True
Action:         English letter(s) detected (10 found) - Translating...
Result:         ಹಲೋ, ಇವತ್ತು ಹೇಗಿದ್ದೀಯ?
----------------------------------------------------------------------------------------------------
[GoogleTranslator] Latency: 396.1 ms | 'What is the capital of India?...' -> 'ಭಾರತದ ರಾಜಧಾನಿ ಯಾವುದು?...'
🔴 TRANSLATED
Type: Pure English
Original:       What is the capital of India?
English found?  True
Action:         English letter(s) detected (10 found) - Translating...
Result:         ಭಾರತದ ರಾಜಧಾನಿ ಯಾವುದು?
----------------------------------------------------------------------------------------------------
[GoogleTranslator] Latency: 352.7 ms | 'Science is very interesting to learn...' -> 'ವಿಜ್ಞಾನವು

In [34]:
# TEST 11: Comprehensive Comparison - Binary Detection (No Confidence Intervals)
comparison_sentences = [
    "Hello, how are you?",
    "I am a student",
    "ನಮಸ್ಕಾರ, ನೀವು ಹೇಗಿದ್ದೀರಿ?",
    "Hello ನಮಸ್ಕಾರ (how are you)?"
]

print("\n" + "=" * 100)
print("TEST 11: COMPREHENSIVE COMPARISON - Binary English Detection")
print("=" * 100)

for sentence in comparison_sentences:
    print(f"\n📝 Input: {sentence}")
    print("-" * 100)
    
    # Method 1: Basic Translation
    print("\n1️⃣  BASIC GoogleTranslator (no optimization):")
    basic_result = translate_to_kannada_google(sentence)
    if basic_result['success']:
        print(f"   ✅ Result: {basic_result['translated'][:70]}")
    else:
        print(f"   ❌ Error: {basic_result['error']}")
    
    # Method 2: Smart with Binary Detection
    print("\n2️⃣  SMART GoogleTranslator (Binary Regex Detection):")
    smart_result = smart_google_translate(sentence)
    print(f"   English found?        {smart_result['has_english_detected']}")
    print(f"   Translation performed? {smart_result['translation_performed']}")
    print(f"   Action:                {smart_result['action_taken']}")
    if smart_result['success']:
        print(f"   ✅ Result: {smart_result['translated'][:70]}")
    else:
        print(f"   ❌ Error: {smart_result.get('error', 'Unknown error')}")
    
    # Method 3: Regex Detection Details
    print("\n3️⃣  Regex English Detection Details:")
    detection = detect_english_with_regex(sentence)
    print(f"   Has English? {detection['has_english']}")
    if detection['english_letters']:
        print(f"   English letters: {detection['english_letters']}")
    
    print("=" * 100)

print("\n✅ Comprehensive comparison complete!")


TEST 11: COMPREHENSIVE COMPARISON - Binary English Detection

📝 Input: Hello, how are you?
----------------------------------------------------------------------------------------------------

1️⃣  BASIC GoogleTranslator (no optimization):
[GoogleTranslator] Latency: 254.3 ms | 'Hello, how are you?...' -> 'ಹಲೋ, ಹೇಗಿದ್ದೀಯಾ?...'
   ✅ Result: ಹಲೋ, ಹೇಗಿದ್ದೀಯಾ?

2️⃣  SMART GoogleTranslator (Binary Regex Detection):
[GoogleTranslator] Latency: 153.5 ms | 'Hello, how are you?...' -> 'ಹಲೋ, ಹೇಗಿದ್ದೀಯಾ?...'
   English found?        True
   Translation performed? True
   Action:                English letter(s) detected (10 found) - Translating...
   ✅ Result: ಹಲೋ, ಹೇಗಿದ್ದೀಯಾ?

3️⃣  Regex English Detection Details:
   Has English? True
   English letters: ['H', 'e', 'l', 'l', 'o', 'h', 'o', 'w', 'a', 'r']

📝 Input: I am a student
----------------------------------------------------------------------------------------------------

1️⃣  BASIC GoogleTranslator (no optimization):
[GoogleTranslator] 

In [35]:
def advanced_english_detection(text):
    """
    Advanced English detection with multiple regex patterns.
    Simple rule: If ANY English letter found in ANY pattern = has_english = True
    
    Detects:
    1. English letters
    2. English words
    3. Contractions (don't, it's)
    4. Capital English words
    5. English abbreviations (e.g., COVID, UNESCO)
    
    Returns detailed breakdown of patterns found (but final answer is binary: True/False).
    """
    analysis = {
        'text': text,
        'has_english': False,
        'patterns': {}
    }
    
    # Pattern 1: Individual English letters [a-zA-Z]
    pattern1 = r'[a-zA-Z]'
    matches1 = re.findall(pattern1, text)
    if matches1:
        analysis['patterns']['english_letters_count'] = len(matches1)
        analysis['has_english'] = True  # BINARY: Found any letter
    
    # Pattern 2: Complete English words \b[a-zA-Z]+\b
    pattern2 = r'\b[a-zA-Z]+\b'
    matches2 = re.findall(pattern2, text)
    if matches2:
        analysis['patterns']['english_words'] = matches2[:5]  # Show first 5
        analysis['has_english'] = True  # BINARY: Found any word
    
    # Pattern 3: Contractions [a-zA-Z]+'[a-zA-Z]+
    pattern3 = r"\b[a-zA-Z]+(?:'[a-zA-Z]+)\b"
    matches3 = re.findall(pattern3, text)
    if matches3:
        analysis['patterns']['contractions'] = matches3
        analysis['has_english'] = True  # BINARY: Found contraction
    
    # Pattern 4: Capital words (proper nouns) [A-Z][a-z]+
    pattern4 = r'\b[A-Z][a-z]+\b'
    matches4 = re.findall(pattern4, text)
    if matches4:
        analysis['patterns']['capital_words'] = matches4[:5]
        analysis['has_english'] = True  # BINARY: Found capital word
    
    # Pattern 5: All caps words/acronyms [A-Z]{2,}
    pattern5 = r'\b[A-Z]{2,}\b'
    matches5 = re.findall(pattern5, text)
    if matches5:
        analysis['patterns']['acronyms'] = matches5
        analysis['has_english'] = True  # BINARY: Found acronym
    
    return analysis

print("✅ Advanced English detection function loaded!")
print("🔍 Detects: letters, words, contractions, capitals, acronyms")
print("💡 Simple rule: Any pattern found = English detected = Translate")

✅ Advanced English detection function loaded!
🔍 Detects: letters, words, contractions, capitals, acronyms
💡 Simple rule: Any pattern found = English detected = Translate


In [36]:
# TEST 12: Advanced English Detection - Pattern Breakdown (Binary: Has English or Not)
advanced_test_cases = [
    "I don't know",  # Contraction
    "Hello India",  # Capital word
    "UNESCO rules",  # Acronym
    "Hello ನಮಸ್ಕಾರ UNESCO ಸಂಸ್ಥೆ",  # Mixed with acronym
    "ನಮಸ್ಕಾರ Dr. Smith",  # Mixed with title and proper noun
    "COVID19 ವೈರಸ್",  # Mixed with acronym-like number
    "ನಾನು don't know ಇಂದು",  # Mixed with contraction
    "ನಮಸ್ಕಾರ! ಹೇಗಿದ್ದೀರಾ? ಯೆಸ್!",  # Mixed with English word at end
    "ನಮಸ್ಕಾರ",  # Pure Kannada (0 English)
]

print("\n" + "=" * 100)
print("TEST 12: ADVANCED ENGLISH DETECTION - Pattern Breakdown (Binary)")
print("=" * 100)

for sentence in advanced_test_cases:
    analysis = advanced_english_detection(sentence)
    
    english_status = "🔴 HAS ENGLISH - WILL TRANSLATE" if analysis['has_english'] else "✅ NO ENGLISH - SKIP"
    print(f"\n{english_status}")
    print(f"📝 Text: {sentence}")
    
    if analysis['patterns']:
        print(f"   Patterns detected:")
        for pattern_name, pattern_data in analysis['patterns'].items():
            if isinstance(pattern_data, list):
                print(f"      • {pattern_name}: {pattern_data}")
            else:
                print(f"      • {pattern_name}: {pattern_data}")
    else:
        print(f"   No English patterns detected")
    
    print("-" * 100)

print("=" * 100)


TEST 12: ADVANCED ENGLISH DETECTION - Pattern Breakdown (Binary)

🔴 HAS ENGLISH - WILL TRANSLATE
📝 Text: I don't know
   Patterns detected:
      • english_letters_count: 9
      • english_words: ['I', 'don', 't', 'know']
      • contractions: ["don't"]
----------------------------------------------------------------------------------------------------

🔴 HAS ENGLISH - WILL TRANSLATE
📝 Text: Hello India
   Patterns detected:
      • english_letters_count: 10
      • english_words: ['Hello', 'India']
      • capital_words: ['Hello', 'India']
----------------------------------------------------------------------------------------------------

🔴 HAS ENGLISH - WILL TRANSLATE
📝 Text: UNESCO rules
   Patterns detected:
      • english_letters_count: 11
      • english_words: ['UNESCO', 'rules']
      • acronyms: ['UNESCO']
----------------------------------------------------------------------------------------------------

🔴 HAS ENGLISH - WILL TRANSLATE
📝 Text: Hello ನಮಸ್ಕಾರ UNESCO ಸಂಸ್ಥೆ
 

In [37]:
# TEST 13: Final Summary and Benchmarking
print("\n" + "=" * 100)
print("TEST 13: SUMMARY - GoogleTranslator with Binary English Detection")
print("=" * 100)

summary_info = {
    'Library': 'deep_translator',
    'Translator': 'GoogleTranslator',
    'Detection Method': 'Binary Regex (ANY English letter = Translate)',
    'Features': [
        '✅ No API keys required',
        '✅ Supports 100+ languages',
        '✅ Free service via Google Translate',
        '✅ Works with mixed language text',
        '✅ Simple binary regex detection [a-zA-Z]',
        '✅ Optimization - skips pure Kannada (0 English letters)',
        '✅ NO confidence intervals - just Yes/No'
    ],
    'Tests Implemented': [
        '✅ TEST 7: Pure English sentences',
        '✅ TEST 8: Mixed English-Kannada',
        '✅ TEST 9: Binary regex detection accuracy',
        '✅ TEST 10: Smart translation with binary detection',
        '✅ TEST 11: Comprehensive method comparison',
        '✅ TEST 12: Advanced pattern breakdown'
    ],
    'Core Logic': [
        'Regex pattern: [a-zA-Z] (any English letter)',
        'IF found ANY letter -> has_english = True -> TRANSLATE',
        'IF found 0 letters -> has_english = False -> SKIP (pure Kannada)',
        'No confidence scoring - simple binary decision',
    ]
}

print("\n📚 IMPLEMENTATION SUMMARY:")
for key, value in summary_info.items():
    if isinstance(value, list):
        print(f"\n{key}:")
        for item in value:
            print(f"   {item}")
    else:
        print(f"\n{key}: {value}")

print("\n" + "=" * 100)
print("🎉 ALL TESTS FOR DEEP_TRANSLATOR GOOGLETRANSLATOR COMPLETE!")
print("💡 Key Rule: Even 1 English letter [a-zA-Z] triggers translation!")
print("=" * 100)


TEST 13: SUMMARY - GoogleTranslator with Binary English Detection

📚 IMPLEMENTATION SUMMARY:

Library: deep_translator

Translator: GoogleTranslator

Detection Method: Binary Regex (ANY English letter = Translate)

Features:
   ✅ No API keys required
   ✅ Supports 100+ languages
   ✅ Free service via Google Translate
   ✅ Works with mixed language text
   ✅ Simple binary regex detection [a-zA-Z]
   ✅ Optimization - skips pure Kannada (0 English letters)
   ✅ NO confidence intervals - just Yes/No

Tests Implemented:
   ✅ TEST 7: Pure English sentences
   ✅ TEST 8: Mixed English-Kannada
   ✅ TEST 9: Binary regex detection accuracy
   ✅ TEST 10: Smart translation with binary detection
   ✅ TEST 11: Comprehensive method comparison
   ✅ TEST 12: Advanced pattern breakdown

Core Logic:
   Regex pattern: [a-zA-Z] (any English letter)
   IF found ANY letter -> has_english = True -> TRANSLATE
   IF found 0 letters -> has_english = False -> SKIP (pure Kannada)
   No confidence scoring - simple 

In [38]:
# TEST 14: Big Paragraph Latency Benchmark — Mixed English + Kannada
# Tests GoogleTranslator (deep_translator) and measures latency per character.

import time

# ── Test paragraphs (mixed English + Kannada) ────────────────────────────
# Paragraph 1: Short (~150 chars)
paragraph_short = (
    "Water is essential for life. "
    "ನೀರು ಜೀವನಕ್ಕೆ ಅವಶ್ಯಕ. "
    "Without water, plants and animals cannot survive."
)

# Paragraph 2: Medium (~350 chars)
paragraph_medium = (
    "Photosynthesis is the process by which green plants make their own food. "
    "ದ್ಯುತಿಸಂಶ್ಲೇಷಣೆ ಎಂದರೆ ಹಸಿರು ಸಸ್ಯಗಳು ಸೂರ್ಯನ ಬೆಳಕಿನ ಸಹಾಯದಿಂದ ತಮ್ಮದೇ ಆಹಾರ ತಯಾರಿಸುವ ಪ್ರಕ್ರಿಯೆ. "
    "They use sunlight, water (H2O), and carbon dioxide (CO2) to produce glucose and oxygen. "
    "ಈ ಪ್ರಕ್ರಿಯೆ ಎಲ್ಲಾ ಜೀವರಾಶಿಗಳಿಗೆ ಆಮ್ಲಜನಕ ಪೂರೈಸುತ್ತದೆ."
)

# Paragraph 3: Long (~700 chars)
paragraph_long = (
    "The human digestive system breaks down food into nutrients that the body can absorb. "
    "ಮಾನವ ಜೀರ್ಣಾಂಗ ವ್ಯವಸ್ಥೆಯು ಆಹಾರವನ್ನು ಪೋಷಕಾಂಶಗಳಾಗಿ ವಿಭಜಿಸುತ್ತದೆ. "
    "The process starts in the mouth where enzymes in saliva begin breaking down carbohydrates. "
    "ಬಾಯಿಯಲ್ಲಿ ಲಾಲಾರಸದ ಕಿಣ್ವಗಳು ಕಾರ್ಬೋಹೈಡ್ರೇಟ್‌ಗಳನ್ನು ಒಡೆಯಲು ಪ್ರಾರಂಭಿಸುತ್ತವೆ. "
    "Food then travels to the stomach where gastric acid and enzymes continue the process. "
    "ಜಠರದಲ್ಲಿ ಗ್ಯಾಸ್ಟ್ರಿಕ್ ಆಮ್ಲ ಮತ್ತು ಕಿಣ್ವಗಳು ಜೀರ್ಣಕ್ರಿಯೆಯನ್ನು ಮುಂದುವರಿಸುತ್ತವೆ. "
    "The small intestine absorbs most nutrients, while the large intestine absorbs water. "
    "ಸಣ್ಣ ಕರುಳು ಹೆಚ್ಚಿನ ಪೋಷಕಾಂಶಗಳನ್ನು ಹೀರಿಕೊಳ್ಳುತ್ತದೆ, ದೊಡ್ಡ ಕರುಳು ನೀರನ್ನು ಹೀರಿಕೊಳ್ಳುತ್ತದೆ."
)

paragraphs = [
    ("Short  (~150 chars)", paragraph_short),
    ("Medium (~350 chars)", paragraph_medium),
    ("Long   (~700 chars)", paragraph_long),
]

# ── Benchmark loop ───────────────────────────────────────────────────────-
print("\n" + "=" * 100)
print("TEST 14: BIG PARAGRAPH LATENCY BENCHMARK — Mixed English + Kannada")
print("Using: deep_translator GoogleTranslator (source=auto, target=kn)")
print("=" * 100)

benchmark_results = []

for label, para in paragraphs:
    char_count = len(para)
    print(f"\n--- {label} | {char_count} chars ---")
    print(f"Input preview : {para[:80]}...")

    # Run 3 trials for a stable estimate
    trial_latencies = []
    translated_text = ""
    for trial in range(1, 4):
        t0 = time.perf_counter()
        translated_text = GoogleTranslator(source='auto', target='kn').translate(para)
        latency_ms = (time.perf_counter() - t0) * 1000
        trial_latencies.append(latency_ms)
        print(f"  Trial {trial}: {latency_ms:.1f} ms")

    avg_ms    = sum(trial_latencies) / len(trial_latencies)
    min_ms    = min(trial_latencies)
    max_ms    = max(trial_latencies)
    ms_per_char = avg_ms / char_count

    print(f"  Output preview: {translated_text[:80]}...")
    print(f"  Avg latency   : {avg_ms:.1f} ms")
    print(f"  Min / Max     : {min_ms:.1f} ms / {max_ms:.1f} ms")
    print(f"  ms per char   : {ms_per_char:.3f} ms/char")

    benchmark_results.append({
        "label"      : label,
        "char_count" : char_count,
        "avg_ms"     : round(avg_ms, 2),
        "min_ms"     : round(min_ms, 2),
        "max_ms"     : round(max_ms, 2),
        "ms_per_char": round(ms_per_char, 4),
        "translated" : translated_text,
    })

print("\n" + "=" * 100)
print("All trials complete. See TEST 15 below for the per-char summary table.")
print("=" * 100)



TEST 14: BIG PARAGRAPH LATENCY BENCHMARK — Mixed English + Kannada
Using: deep_translator GoogleTranslator (source=auto, target=kn)

--- Short  (~150 chars) | 100 chars ---
Input preview : Water is essential for life. ನೀರು ಜೀವನಕ್ಕೆ ಅವಶ್ಯಕ. Without water, plants and ani...
  Trial 1: 689.1 ms
  Trial 2: 225.2 ms
  Trial 3: 184.6 ms
  Output preview: ಜೀವನಕ್ಕೆ ನೀರು ಅತ್ಯಗತ್ಯ. ನೀರು ಜೀವನಕ್ಕೆ ಅವಶ್ಯಕ. ನೀರಿಲ್ಲದೆ, ಸಸ್ಯಗಳು ಮತ್ತು ಪ್ರಾಣಿಗಳು...
  Avg latency   : 366.3 ms
  Min / Max     : 184.6 ms / 689.1 ms
  ms per char   : 3.663 ms/char

--- Medium (~350 chars) | 303 chars ---
Input preview : Photosynthesis is the process by which green plants make their own food. ದ್ಯುತಿಸ...
  Trial 1: 312.7 ms
  Trial 2: 176.1 ms
  Trial 3: 172.7 ms
  Output preview: Photosynthesis is the process by which green plants make their own food. ದ್ಯುತಿಸ...
  Avg latency   : 220.5 ms
  Min / Max     : 172.7 ms / 312.7 ms
  ms per char   : 0.728 ms/char

--- Long   (~700 chars) | 644 chars ---
Input preview : The human 

In [41]:
# TEST 15: Per-Character Latency Summary & Reportable Metrics
# Summarises results from TEST 14 — run TEST 14 first.

print("\n" + "=" * 100)
print("TEST 15: PER-CHARACTER LATENCY SUMMARY (GoogleTranslator / deep_translator)")
print("=" * 100)

# ─── Table header ──────────────────────────────────────────────────────────
col_w = [22, 8, 10, 10, 10, 14]
headers = ["Paragraph", "Chars", "Avg (ms)", "Min (ms)", "Max (ms)", "ms / char"]
header_row = "  ".join(h.ljust(w) for h, w in zip(headers, col_w))
print("\n" + header_row)
print("-" * len(header_row))

for r in benchmark_results:
    row = "  ".join([
        r["label"].ljust(col_w[0]),
        str(r["char_count"]).ljust(col_w[1]),
        f"{r['avg_ms']:.1f}".ljust(col_w[2]),
        f"{r['min_ms']:.1f}".ljust(col_w[3]),
        f"{r['max_ms']:.1f}".ljust(col_w[4]),
        f"{r['ms_per_char']:.4f}".ljust(col_w[5]),
    ])
    print(row)

print("-" * len(header_row))

# ─── Overall weighted average ms/char ──────────────────────────────────────
total_chars = sum(r["char_count"] for r in benchmark_results)
total_weighted_ms = sum(r["avg_ms"] * r["char_count"] for r in benchmark_results)
overall_ms_per_char = total_weighted_ms / total_chars
overall_avg_ms = sum(r["avg_ms"] for r in benchmark_results) / len(benchmark_results)

print()
print(f"  Overall weighted avg ms/char : {overall_ms_per_char:.4f} ms/char")
print(f"  Overall simple avg latency   : {overall_avg_ms:.1f} ms  (across all 3 paragraph sizes)")

# ─── Interpretable summary ─────────────────────────────────────────────────
print("\n" + "=" * 100)
print("REPORTABLE SUMMARY")
print("=" * 100)
print(f"""
Library     : deep_translator (GoogleTranslator)
Direction   : auto -> Kannada (kn)
Text type   : Mixed English + Kannada paragraphs

Latency (3-trial avg per paragraph size):
  Short  paragraph (~150 chars)  : {benchmark_results[0]['avg_ms']:.0f} ms
  Medium paragraph (~350 chars)  : {benchmark_results[1]['avg_ms']:.0f} ms
  Long   paragraph (~700 chars)  : {benchmark_results[2]['avg_ms']:.0f} ms

Per-character latency:
  Short  : {benchmark_results[0]['ms_per_char']:.4f} ms/char
  Medium : {benchmark_results[1]['ms_per_char']:.4f} ms/char
  Long   : {benchmark_results[2]['ms_per_char']:.4f} ms/char
  Weighted avg across all sizes: {overall_ms_per_char:.4f} ms/char

""")
print("=" * 100)



TEST 15: PER-CHARACTER LATENCY SUMMARY (GoogleTranslator / deep_translator)

Paragraph               Chars     Avg (ms)    Min (ms)    Max (ms)    ms / char     
------------------------------------------------------------------------------------
Short  (~150 chars)     100       366.3       184.6       689.1       3.6630        
Medium (~350 chars)     303       220.5       172.7       312.7       0.7277        
Long   (~700 chars)     644       277.2       186.9       326.2       0.4305        
------------------------------------------------------------------------------------

  Overall weighted avg ms/char : 269.3138 ms/char
  Overall simple avg latency   : 288.0 ms  (across all 3 paragraph sizes)

REPORTABLE SUMMARY

Library     : deep_translator (GoogleTranslator)
Direction   : auto -> Kannada (kn)
Text type   : Mixed English + Kannada paragraphs

Latency (3-trial avg per paragraph size):
  Short  paragraph (~150 chars)  : 366 ms
  Medium paragraph (~350 chars)  : 220 ms
  Long